In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt


# --------------------------------------------------
# 1. Load and preprocess MNIST dataset
# --------------------------------------------------

(x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()

# Normalize images from [0, 255] to [-1, 1]
x_train = x_train.astype("float32")
x_train = (x_train - 127.5) / 127.5

# Add channel dimension: (28, 28) -> (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)

# Create batches
BUFFER_SIZE = 60000
BATCH_SIZE = 256

dataset = tf.data.Dataset.from_tensor_slices(x_train)
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE)


# --------------------------------------------------
# 2. Create the Generator
# --------------------------------------------------

def create_generator():
    model = tf.keras.Sequential([
        layers.Dense(
            7 * 7 * 256,
            use_bias=False,
            input_shape=(100,)
        ),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Reshape((7, 7, 256)),

        layers.Conv2DTranspose(
            128,
            (5, 5),
            strides=(1, 1),
            padding="same",
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(
            64,
            (5, 5),
            strides=(2, 2),
            padding="same",
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(
            1,
            (5, 5),
            strides=(2, 2),
            padding="same",
            use_bias=False,
            activation="tanh"
        )
    ])

    return model


# --------------------------------------------------
# 3. Create the Discriminator
# --------------------------------------------------

def create_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(
            64,
            (5, 5),
            strides=(2, 2),
            padding="same",
            input_shape=[28, 28, 1]
        ),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(
            128,
            (5, 5),
            strides=(2, 2),
            padding="same"
        ),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])

    return model


generator = create_generator()
discriminator = create_discriminator()


# --------------------------------------------------
# 4. Define Loss Function
# --------------------------------------------------

cross_entropy = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)


def generator_loss(fake_output):
    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )


def discriminator_loss(real_output, fake_output):

    real_loss = cross_entropy(
        tf.ones_like(real_output),
        real_output
    )

    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )

    return real_loss + fake_loss


# --------------------------------------------------
# 5. Define Optimizers
# --------------------------------------------------

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)


# --------------------------------------------------
# 6. Training Function
# --------------------------------------------------

@tf.function
def train_step(images):

    # Use the actual batch size
    batch_size = tf.shape(images)[0]

    noise = tf.random.normal(
        [batch_size, 100]
    )

    with tf.GradientTape() as gen_tape, \
         tf.GradientTape() as disc_tape:

        # Generate fake images
        generated_images = generator(
            noise,
            training=True
        )

        # Discriminator predictions
        real_output = discriminator(
            images,
            training=True
        )

        fake_output = discriminator(
            generated_images,
            training=True
        )

        # Calculate losses
        gen_loss = generator_loss(fake_output)

        disc_loss = discriminator_loss(
            real_output,
            fake_output
        )

    # Calculate gradients
    gradients_of_generator = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    gradients_of_discriminator = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    # Update networks
    generator_optimizer.apply_gradients(
        zip(
            gradients_of_generator,
            generator.trainable_variables
        )
    )

    discriminator_optimizer.apply_gradients(
        zip(
            gradients_of_discriminator,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss


# --------------------------------------------------
# 7. Generate Sample Images
# --------------------------------------------------

seed = tf.random.normal([16, 100])


def generate_images(epoch):

    predictions = generator(
        seed,
        training=False
    )

    plt.figure(figsize=(4, 4))

    for i in range(16):

        plt.subplot(4, 4, i + 1)

        plt.imshow(
            predictions[i, :, :, 0] * 127.5 + 127.5,
            cmap="gray"
        )

        plt.axis("off")

    plt.suptitle(
        "Generated Images - Epoch " + str(epoch)
    )

    plt.show()


# --------------------------------------------------
# 8. Train GAN
# --------------------------------------------------

EPOCHS = 30

for epoch in range(EPOCHS):

    total_gen_loss = 0.0
    total_disc_loss = 0.0

    for image_batch in dataset:

        gen_loss, disc_loss = train_step(
            image_batch
        )

        total_gen_loss += float(gen_loss)
        total_disc_loss += float(disc_loss)

    print(
        "Epoch:",
        epoch + 1,
        "Generator Loss:",
        total_gen_loss,
        "Discriminator Loss:",
        total_disc_loss
    )

    # Display generated images every 5 epochs
    if (epoch + 1) % 5 == 0:
        generate_images(epoch + 1)


# --------------------------------------------------
# 9. Generate Final Images
# --------------------------------------------------

generate_images("Final")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch: 1 Generator Loss: 159.33886888623238 Discriminator Loss: 302.3519517183304
Epoch: 2 Generator Loss: 174.24595201015472 Discriminator Loss: 314.74446189403534
Epoch: 3 Generator Loss: 174.58584839105606 Discriminator Loss: 317.6996001005173
